# 02 - Data Cleaning

## Goal

Clean and validate the raw Ticketmaster event data collected during API ingestion.

## Tasks

- Load the raw JSON data
- Flatten nested event data
- Investigate duplicate events
- Handle missing values
- Validate dates and times
- Convert geographic fields to correct types
- Standardize text fields
- Save the cleaned dataset

In [1]:
import pandas as pd
import json
from pathlib import Path

In [2]:
project_path = Path("..")
raw_path = (
    project_path / "data" / "raw" / "ticketmaster" / "germany_music_events.json"
    )

with open(raw_path, "r", encoding= "utf-8") as file:
    raw_events = json.load(file)

In [9]:
type(raw_events)
len(raw_events)

list

## Flatten Raw Events

The Ticketmaster API returns nested JSON data.  
The raw events are flattened into a tabular structure for cleaning and validation.

In [5]:
def flatten_event(event):
    embedded = event.get("_embedded", {})

    venues = embedded.get("venues", [])
    venue = venues[0] if venues else {}

    attractions = embedded.get("attractions", [])
    artist = attractions[0] if attractions else {}

    location = venue.get("location", {})

    return {
        "event_id": event.get("id"),
        "event_name": event.get("name"),
        "artist_name": artist.get("name"),
        "event_date": event.get("dates", {}).get("start", {}).get("localDate"),
        "event_time": event.get("dates", {}).get("start", {}).get("localTime"),
        "venue_name": venue.get("name"),
        "city": venue.get("city", {}).get("name"),
        "country": venue.get("country", {}).get("name"),
        "latitude": location.get("latitude"),
        "longitude": location.get("longitude"),
        "event_url": event.get("url")
    }
    

In [6]:
flatten_event(raw_events[0
])

{'event_id': 'LvZ18QLUFcKuwNYZ0XXWn',
 'event_name': 'Heavysaurus - METAL Tour 2026',
 'artist_name': 'Heavysaurus',
 'event_date': '2026-08-30',
 'event_time': '13:00:00',
 'venue_name': 'Schön & Frölich',
 'city': 'Braunschweig',
 'country': 'Germany',
 'latitude': '52.25654',
 'longitude': '10.49957',
 'event_url': 'https://www.universe.com/events/heavysaurus-metal-tour-2026-tickets-X4HCW6?ref=ticketmaster'}

In [7]:
rows = [
    flatten_event(event)
    for event in raw_events
]

events_df = pd.DataFrame(rows)

In [15]:
events_df.head()

,event_id,event_name,artist_name,event_date,event_time,venue_name,city,country,latitude,longitude,event_url
0,LvZ18QLUFcKuwNYZ0XXWn,Heavysaurus - METAL Tour 2026,Heavysaurus,2026-08-30,13:00:00,Schön & Frölich,Braunschweig,Germany,52.25654,10.49957,https://www.universe.com/events/heavysaurus-me...
1,Z698xZC2Z16v8KeAfJ,Melanie Martinez – HADES: THE SACRIFICE | VIP,Melanie Martinez,2026-09-18,20:00:00,NaN,Frankfurt am Main,Germany,50.11233,8.65073,https://www.ticketmaster.de/event/melanie-mart...
2,Z698xZC2Z1kAIGIZg,Melanie Martinez – HADES: THE SACRIFICE,Melanie Martinez,2026-09-18,20:00:00,NaN,Frankfurt am Main,Germany,50.11233,8.65073,https://www.ticketmaster.de/event/melanie-mart...
3,LvZ18Qpz8oKu0POZyE61A,SUPERBLOOM 2026 Experience Day - Sonntag,SUPERBLOOM Festival,2026-08-30,10:00:00,NaN,Munich,Germany,48.17429,11.55524,https://www.universe.com/events/superbloom-202...
4,Z698xZC2Z1kCpjv8P,ITZY 3RD WORLD TOUR <TUNNEL VISION> in FRANKFURT,ITZY,2026-09-17,19:30:00,NaN,Frankfurt am Main,Germany,50.11233,8.65073,https://www.ticketmaster.de/event/itzy-3rd-wor...


## Duplicate Events

Duplicate event IDs are investigated before removal to ensure that repeated records do not contain conflicting information.

In [20]:
duplicate_events = events_df[
    events_df["event_id"].duplicated(keep= False)
].sort_values("event_id")

duplicate_events

,event_id,event_name,artist_name,event_date,event_time,venue_name,city,country,latitude,longitude,event_url
943,LvZ18QNf90GZCmOvG95vu,FRITZ KALKBRENNER,Fritz Kalkbrenner,2026-12-19,22:00:00,KARREE,Freiburg,Germany,47.99808,7.84851,https://www.universe.com/events/fritz-kalkbren...
852,LvZ18QNf90GZCmOvG95vu,FRITZ KALKBRENNER,Fritz Kalkbrenner,2026-12-19,22:00:00,KARREE,Freiburg,Germany,47.99808,7.84851,https://www.universe.com/events/fritz-kalkbren...
1059,LvZ18QQLjjSI3-YZ0gwJ5,BOND NIGHT mit Dennis Durant & Band im Hotel A...,DWEDA Records,2027-02-19,19:00:00,NaN,Hamburg,Germany,53.55751,10.00524,https://www.universe.com/events/bond-night-mit...
1013,LvZ18QQLjjSI3-YZ0gwJ5,BOND NIGHT mit Dennis Durant & Band im Hotel A...,DWEDA Records,2027-02-19,19:00:00,NaN,Hamburg,Germany,53.55751,10.00524,https://www.universe.com/events/bond-night-mit...
491,Z698xZC2Z16v4fG4uE,Hatsune Miku - MIKU EXPO 2026 EUROPE,Hatsune Miku,2026-11-17,20:00:00,NaN,Berlin,Germany,52.53117,13.45067,https://www.ticketmaster.de/event/hatsune-miku...
508,Z698xZC2Z16v4fG4uE,Hatsune Miku - MIKU EXPO 2026 EUROPE,Hatsune Miku,2026-11-17,20:00:00,NaN,Berlin,Germany,52.53117,13.45067,https://www.ticketmaster.de/event/hatsune-miku...
981,Z698xZC2Z16v4s8xP4,Unheilig | Box seat in the Ticketmaster Suite,Unheilig,2027-01-29,19:45:00,Barclays Arena,Hamburg,Germany,53.58894,9.899,https://www.ticketmaster.de/event/unheilig-%7C...
983,Z698xZC2Z16v4s8xP4,Unheilig | Box seat in the Ticketmaster Suite,Unheilig,2027-01-29,19:45:00,Barclays Arena,Hamburg,Germany,53.58894,9.899,https://www.ticketmaster.de/event/unheilig-%7C...
222,Z698xZC2Z16vOyPJvp,Bryan Adams - Roll With The Punches Tour,Bryan Adams,2026-10-12,20:00:00,SAP Arena,Mannheim,Germany,49.46424,8.51797,https://www.ticketmaster.de/event/bryan-adams-...
217,Z698xZC2Z16vOyPJvp,Bryan Adams - Roll With The Punches Tour,Bryan Adams,2026-10-12,20:00:00,SAP Arena,Mannheim,Germany,49.46424,8.51797,https://www.ticketmaster.de/event/bryan-adams-...


In [23]:
duplicate_events["event_id"].nunique()

9

In [ ]:
duplicate_comparison = (
    duplicate_events.groupby("event_id").nunique(dropna= False)
)
duplicate_comparison

In [33]:
events_df = (
    events_df.drop_duplicates(subset= "event_id", keep= "first").reset_index(drop= True)
)

events_df.head()

,event_id,event_name,artist_name,event_date,event_time,venue_name,city,country,latitude,longitude,event_url
0,LvZ18QLUFcKuwNYZ0XXWn,Heavysaurus - METAL Tour 2026,Heavysaurus,2026-08-30,13:00:00,Schön & Frölich,Braunschweig,Germany,52.25654,10.49957,https://www.universe.com/events/heavysaurus-me...
1,Z698xZC2Z16v8KeAfJ,Melanie Martinez – HADES: THE SACRIFICE | VIP,Melanie Martinez,2026-09-18,20:00:00,NaN,Frankfurt am Main,Germany,50.11233,8.65073,https://www.ticketmaster.de/event/melanie-mart...
2,Z698xZC2Z1kAIGIZg,Melanie Martinez – HADES: THE SACRIFICE,Melanie Martinez,2026-09-18,20:00:00,NaN,Frankfurt am Main,Germany,50.11233,8.65073,https://www.ticketmaster.de/event/melanie-mart...
3,LvZ18Qpz8oKu0POZyE61A,SUPERBLOOM 2026 Experience Day - Sonntag,SUPERBLOOM Festival,2026-08-30,10:00:00,NaN,Munich,Germany,48.17429,11.55524,https://www.universe.com/events/superbloom-202...
4,Z698xZC2Z1kCpjv8P,ITZY 3RD WORLD TOUR <TUNNEL VISION> in FRANKFURT,ITZY,2026-09-17,19:30:00,NaN,Frankfurt am Main,Germany,50.11233,8.65073,https://www.ticketmaster.de/event/itzy-3rd-wor...


In [34]:
print("Rows after duplicate removal:", len(events_df))
print("Duplicate event IDs:", events_df["event_id"].duplicated().sum())
print("Event IDs are unique:", events_df["event_id"].is_unique)

Rows after duplicate removal: 1172
Duplicate event IDs: 0
Event IDs are unique: True


## Missing Values

Missing values are reviewed to determine whether records should be retained, corrected, or removed.

In [36]:
missing_summary = pd.DataFrame({
    "missing_count":events_df.isna().sum(),
    "missing_percent": events_df.isna().mean() * 100
})

missing_summary[missing_summary["missing_count"] > 0].sort_values("missing_count", ascending= False)

,missing_count,missing_percent
venue_name,689,58.788396
event_time,9,0.767918
artist_name,3,0.255973


In [40]:
missing_venue = events_df[events_df["venue_name"].isna()]

missing_venue.head()

,event_id,event_name,artist_name,event_date,event_time,venue_name,city,country,latitude,longitude,event_url
1,Z698xZC2Z16v8KeAfJ,Melanie Martinez – HADES: THE SACRIFICE | VIP,Melanie Martinez,2026-09-18,20:00:00,NaN,Frankfurt am Main,Germany,50.11233,8.65073,https://www.ticketmaster.de/event/melanie-mart...
2,Z698xZC2Z1kAIGIZg,Melanie Martinez – HADES: THE SACRIFICE,Melanie Martinez,2026-09-18,20:00:00,NaN,Frankfurt am Main,Germany,50.11233,8.65073,https://www.ticketmaster.de/event/melanie-mart...
3,LvZ18Qpz8oKu0POZyE61A,SUPERBLOOM 2026 Experience Day - Sonntag,SUPERBLOOM Festival,2026-08-30,10:00:00,NaN,Munich,Germany,48.17429,11.55524,https://www.universe.com/events/superbloom-202...
4,Z698xZC2Z1kCpjv8P,ITZY 3RD WORLD TOUR <TUNNEL VISION> in FRANKFURT,ITZY,2026-09-17,19:30:00,NaN,Frankfurt am Main,Germany,50.11233,8.65073,https://www.ticketmaster.de/event/itzy-3rd-wor...
5,Z698xZC2Z16vZJQ3-g,RAWAYANA - ¿Dónde Es El After? World Tour,Rawayana,2026-09-14,20:00:00,NaN,Berlin,Germany,52.48461,13.3914,https://www.ticketmaster.de/event/rawayana-don...


In [41]:
missing_venue.keys()

Index(['event_id', 'event_name', 'artist_name', 'event_date', 'event_time',
       'venue_name', 'city', 'country', 'latitude', 'longitude', 'event_url'],
      dtype='str')

In [43]:
missing_venue[["city", "country", "latitude", "longitude"]].isna().sum()

city         0
country      0
latitude     0
longitude    0
dtype: int64

In [45]:
missing_artist = events_df[events_df["artist_name"].isna()]

missing_artist

,event_id,event_name,artist_name,event_date,event_time,venue_name,city,country,latitude,longitude,event_url
235,Z698xZC2Z1kx_Obx8,Martinique,NaN,2026-10-10,19:00:00,Kulturzentrum Murkens Hof,Lilienthal,Germany,53.1409,8.91381,https://www.ticketmaster.de/event/martinique-t...
948,Z698xZC2Z16v0KfF8f,TY FREEMAN,NaN,2027-01-07,21:00:00,NaN,Hamburg,Germany,53.55789,9.9679,https://www.ticketmaster.de/event/ty-freeman-t...
1053,Z698xZC2Z1kJvAJFO,Union 1850 & Lucky L & lupus - STAMMTISCH RALL...,NaN,2027-02-25,20:00:00,NaN,Leipzig,Germany,51.33254,12.33841,https://www.ticketmaster.de/event/union-1850--...


In [47]:
missing_time = events_df[events_df["event_time"].isna()]

missing_time

,event_id,event_name,artist_name,event_date,event_time,venue_name,city,country,latitude,longitude,event_url
930,Z698xZC2Z1k3o13fN,JOHNO37 - Tour 2026 | Premium Upgrade (no Tick...,Johno37,2026-12-07,NaN,NaN,Frankfurt am Main,Germany,50.13557,8.73896,https://www.ticketmaster.de/event/johno37-tour...
931,Z698xZC2Z1kaf1Avb,JOHNO37 - Tour 2026 | Premium Upgrade (no Tick...,Johno37,2026-12-08,NaN,NaN,Munich,Germany,48.14736,11.52037,https://www.ticketmaster.de/event/johno37-tour...
932,Z698xZC2Z16vxK46rk,JOHNO37 - Tour 2026 | Premium Upgrade (no Tick...,Johno37,2026-12-09,NaN,NaN,Berlin,Germany,52.46438,13.43338,https://www.ticketmaster.de/event/johno37-tour...
933,Z698xZC2Z1kG8xdOp,JOHNO37 - Tour 2026 | Premium Upgrade (no Tick...,Johno37,2026-12-11,NaN,NaN,Stuttgart,Germany,48.77504,9.17705,https://www.ticketmaster.de/event/johno37-tour...
934,Z698xZC2Z1kvEqgra,JOHNO37 - Tour 2026 | Premium Upgrade (no Tick...,Johno37,2026-12-14,NaN,NaN,Hamburg,Germany,53.55119,9.95788,https://www.ticketmaster.de/event/johno37-tour...
943,Z698xZC2Z16evwpe7N,Kanii,Kanii,2026-12-31,NaN,NaN,Cologne,Germany,50.95023,6.91347,https://www.ticketmaster.de/event/kanii-ticket...
944,Z698xZC2Z1k-_bdFZ,Kanii,Kanii,2026-12-31,NaN,NaN,Berlin,Germany,52.50833,13.45493,https://www.ticketmaster.de/event/kanii-ticket...
1078,Z698xZC2Z16vr3epZy,TMF - Trier Music Festival 2025,TMF Trier Music Festival,2027-03-20,NaN,Messepark,Trier,Germany,49.74055,6.62215,https://www.ticketmaster.de/event/tmf-trier-mu...
1171,Z698xZC2Z16vFVG77P,MAGMA 2027 | 07.08. & 08.08.2027 | WEEKEND TICKET,MAGMA Festival,2027-08-07,NaN,RSO.BERLIN,Berlin,Germany,52.46054,13.50445,https://www.ticketmaster.de/event/magma-2027-%...


- Missing values were preserved as null values rather than replaced with artificial placeholders.

In [55]:
events_df.dtypes

event_id                  str
event_name                str
artist_name               str
event_date     datetime64[us]
event_time                str
venue_name                str
city                      str
country                   str
latitude              float64
longitude             float64
event_url                 str
dtype: object

In [ ]:
events_df["event_date"] = pd.to_datetime(events_df["event_date"], errors= "coerce")
events_df["event_date"].isna().sum()

In [25]:
events_df["latitude"] = pd.to_numeric(events_df["latitude"], errors= "coerce")
events_df["longitude"] = pd.to_numeric(events_df["longitude"], errors= "coerce")

In [26]:
invalid_coordinates = events_df[
    ~events_df["latitude"].between(-90, 90) | ~events_df["longitude"].between(-180, 180)
]

invalid_coordinates

,event_id,event_name,artist_name,event_date,event_time,venue_name,city,country,latitude,longitude,event_url


## Event Time Validation

Event times are validated and converted to time values while preserving missing times.

In [ ]:
parsed_time = pd.to_datetime(events_df["event_time"],
                             format= "%H:%M:%S",
                             errors="coerce")


pandas.Series

In [13]:
invalid_time = events_df[events_df["event_time"].notna() & parsed_time.isna()]
invalid_time

,event_id,event_name,artist_name,event_date,event_time,venue_name,city,country,latitude,longitude,event_url


In [14]:
events_df["event_time"] = parsed_time.dt.time

In [15]:
events_df["event_date"].agg(["min", "max"])

min    2026-08-20
max    2027-08-07
Name: event_date, dtype: str

In [16]:
events_df["event_date"].isna().sum()

np.int64(0)

## Text Standardization

Text fields are standardized by removing unnecessary leading and trailing whitespace while preserving missing values.

In [18]:
text_columns =[
    "event_name",
    "artist_name",
    "venue_name",
    "city",
    "country"
]

for column in text_columns:
    events_df[column] = events_df[column].str.strip()

In [19]:
events_df[text_columns] = events_df[text_columns].replace(r"^\s*$", pd.NA, regex= True)

In [20]:
events_df["country"].value_counts(dropna=False)

country
Germany    1181
Name: count, dtype: int64

## Final Validation

The cleaned dataset is validated before being saved for downstream database loading and geospatial analysis.

In [27]:
print("Rows:", len(events_df))
print("Columns:", events_df.shape[1])

print("\nData integrity:")
print("Missing event IDs:", events_df["event_id"].isna().sum())
print("Duplicate event IDs:", events_df["event_id"].duplicated().sum())
print("Event IDs are unique:", events_df["event_id"].is_unique)

print("\nDates:")
print("Missing event dates:", events_df["event_date"].isna().sum())
print("Earliest event:", events_df["event_date"].min())
print("Latest event:", events_df["event_date"].max())

print("\nGeography:")
print("Missing latitude:", events_df["latitude"].isna().sum())
print("Missing longitude:", events_df["longitude"].isna().sum())

Rows: 1181
Columns: 11

Data integrity:
Missing event IDs: 0
Duplicate event IDs: 9
Event IDs are unique: False

Dates:
Missing event dates: 0
Earliest event: 2026-08-20
Latest event: 2027-08-07

Geography:
Missing latitude: 0
Missing longitude: 0


In [ ]:
processed_path = project_path / "data" / "processed"
processed_path.mkdir(parents= True, exist_ok=True)

clean_file = processed_path / "ticketmaster_events_clean.csv"

events_df.to_csv(clean_file, index= False)

In [34]:
clean_file

WindowsPath('../data/processed/ticketmaster_events_clean.csv')